# 🕵️ Case file 02 — The answer the booking system couldn't read

> **Mission briefing.** Same chaotic day, new incident. The booking system keeps rejecting what the assistant sends it. The next reader is software, not a person: it needs exact fields with the right types before it can book anything. Open the activity logs and find out why the answers are rejected, then make the assistant produce answers the system can use.

Each run leaves an **activity log** (Langfuse calls it a *trace*), made of **steps** (*spans*). The step where the assistant writes its answer is called a *generation*. Its activity log shows the instructions the assistant received and the result of the booking system's check.

## Case objective
Find out what makes an answer usable for the next system, and change the instructions until every answer is.

## What to expect
Three levels. Each run prints whether the booking system accepted the answer, plus a link to the activity log. The reason codes and details are only in the activity log.

## Bad practice
Asking for whatever answer the assistant likes and hoping the next system can read it.

## Good practice
Agree on an exact format with the next system, ask for it explicitly, and check every answer before using it.

## Why the good practice works
A checkable format turns a vague failure into a clear reason, so bad data is stopped before it does damage.

## Deliverable
A claim code for each level you solve, and for your demo, your final instructions and the five-request activity log.

## 🎮 How the game works

- Put your team name in the next cell and run it. Use **the same name in every notebook**.
- Each level asks you to find something in the activity logs. Type your answer in `check_flag(...)` and run it. Wrong guesses cost nothing.
- Correct answers give a **claim code**. Paste it into the game master's form.
- Stuck? `hint("2-1")` shows a hint. Each hint costs 25% of that level's points (two hints at most).
- Points: Level 1 = 100, Level 2 = 200, Level 3 = 300.

In [1]:
from experiment_utils import (STRUCTURED_INSTRUCTIONS, check_flag, close_case, hint, run_format,
                              run_format_batch, set_team, show_batch_results, show_format_results)

TEAM = "your-team-name"  # ← your team name, the same in every notebook
set_team(TEAM)

⚠️  TEAM is still 'your-team-name'. Claim codes will not score: change it and rerun this cell.


'your-team-name'

## Level 1 — The prose problem (100 points)

The same booking request goes to the assistant twice: once asking for a one-sentence summary, once asking for exact fields.

**Find:** the reason code the booking system recorded for the free-text answer. It is in the activity log's result, written in capitals.

In [2]:
runs = [run_format("unconstrained"), run_format("constrained")]
show_format_results(runs)


level 1 · free text
  answer: The customer requests a booking for 2 passengers in economy class from Lisbon to Madrid on December 1, 2026.
  booking system: REJECTED — the answer is not a structured object
  tokens: 60 in / 66 out · latency: 1.1s
  activity log: http://localhost:3000/project/langfuse-observability/traces/5e9d7aa680c837e628b27a0abe0ff451

level 1 · structured
  answer: {"origin":"Lisbon","destination":"Madrid","travel_date":"2026-12-01","passengers":2,"cabin":"economy"}
  booking system: ACCEPTED — all fields present and correct
  tokens: 102 in / 65 out · latency: 0.9s
  activity log: http://localhost:3000/project/langfuse-observability/traces/d9858e00726b690ad9592c4a8594513a


In [3]:
check_flag("2-1", "type your answer here")
# hint("2-1")

❌ Not quite. Try again, or run hint("2-1").


## Level 2 — Five messy requests (200 points)

Real customers do not write tidy requests. Five of them go through the structured instructions. Some answers get rejected, and some may get accepted when they should not be.

**Find:** for request 3, the one field of the booking format that cannot hold what the customer asked for.

In [4]:
batch = run_format_batch(STRUCTURED_INSTRUCTIONS, 2)
show_batch_results(batch)


level 2 · 4 of 5 answers usable
  ✅ request 1: Book 2 passengers from Lisbon to Madrid on 2026-12-01, economy.
  ✅ request 2: Me and my partner want to fly Porto to Paris next Friday, cheapest seats.
  ✅ request 3: One seat Lisbon to London on 2026-12-10 in business class.
  ❌ request 4: Two adults to Rome on 2026-12-15, premium.
  ✅ request 5: Book three people Madrid to Berlin on 15/12/2026, economy.
  activity log: http://localhost:3000/project/langfuse-observability/traces/1758b17dd7de66a34603eae70d737a79


In [5]:
check_flag("2-2", "type your answer here")
# hint("2-2")

❌ Not quite. Try again, or run hint("2-2").


## Level 3 — Make the booking system happy (300 points)

Rewrite the instructions so that **all five** answers are usable:

- requests that contain everything become a booking the system accepts;
- requests with a missing or unsupported detail become `{"needs_clarification": "<field>"}`, naming the field the assistant must ask the customer about.

Edit `my_instructions`, run both cells, and read each request's step in the activity log to see why it failed. When all five are usable, the notebook prints a flag.

In [6]:
my_instructions = STRUCTURED_INSTRUCTIONS  # ← replace with your own instructions (a text in quotes)

In [7]:
level3 = run_format_batch(my_instructions, 3)
show_batch_results(level3)


level 3 · 2 of 5 answers usable
  ✅ request 1: Book 2 passengers from Lisbon to Madrid on 2026-12-01, economy.
  ❌ request 2: Me and my partner want to fly Porto to Paris next Friday, cheapest seats.
  ❌ request 3: One seat Lisbon to London on 2026-12-10 in business class.
  ❌ request 4: Two adults to Rome on 2026-12-15, premium.
  ✅ request 5: Book three people Madrid to Berlin on 15/12/2026, economy.
  activity log: http://localhost:3000/project/langfuse-observability/traces/e185738e6cd799a21d16ae2c19401641


In [8]:
check_flag("2-3", "type your answer here")
# hint("2-3")

❌ Not quite. Try again, or run hint("2-3").


## 🏁 Wrap-up

Run the last cell to close the case and send the activity logs to Langfuse.

In [9]:
close_case("activity-2", runs + [batch, level3])


case activity-2 closed — session activity-2-01b8616d
level 1 · free text: http://localhost:3000/project/langfuse-observability/traces/5e9d7aa680c837e628b27a0abe0ff451
level 1 · structured: http://localhost:3000/project/langfuse-observability/traces/d9858e00726b690ad9592c4a8594513a
level 2 · five requests, 4 usable: http://localhost:3000/project/langfuse-observability/traces/1758b17dd7de66a34603eae70d737a79
level 3 · five requests, 2 usable: http://localhost:3000/project/langfuse-observability/traces/e185738e6cd799a21d16ae2c19401641
activity logs flushed to Langfuse
